# Wildfire Mitigation Multi-Agent Simulation — Stage 1 Prototype
## Single Agent, Single Event

**Creator:** [Your team names here]  
**Course:** INFO 290: Fundamentals of Generative AI (Spring 2026)  
**Instructor:** Cornelia Paulik

#### ``Citations``
- Park, J.S. et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* arXiv:2304.03442
- Notebook structure adapted from course studio notebooks (Cornelia Paulik, INFO 290)

#### ``Learning objectives``

1. Load and initialise a single agent (Margaret) from a structured YAML config containing her persona, seed memories, and held-out interview responses.

2. Implement the perceive → retrieve → decide → store cognition cycle for one agent receiving one intervention.

3. Verify in-character behaviour: does Margaret's simulated response match what she said in the real interview?

#### ``Stage 1 scope``
- One agent: Margaret (The Resourceful DIYer)
- One hardcoded event: insurance non-renewal notice (official mail channel)
- No retrieval scoring yet — all seed memories passed to LLM (retrieval.py added in Stage 2)
- No simulation loop, scheduler, or multi-agent interactions yet

---
### Step 1: Install libraries

In [ ]:
!pip install -U -q --disable-pip-version-check --no-warn-script-location \
  anthropic \
  openai \
  numpy \
  pyyaml

---
### Step 2: Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Update this path to wherever you cloned the repo in your Drive
PROJECT_PATH = "/content/drive/MyDrive/wildfire-sim"
os.chdir(PROJECT_PATH)
!pwd
!ls $PROJECT_PATH

---
### Step 3: Import libraries

In [ ]:
import sys
import json
import yaml
from pathlib import Path
from datetime import datetime

# Add project root to path so we can import src modules
sys.path.insert(0, PROJECT_PATH)

from src.llm.client import Config, init_clients, decide, embed, score_importance, reflect
from src.agents.memory import MemoryStream
from src.agents.prompts import (
    build_system_prompt,
    build_decision_prompt,
    frame_event,
    DECISION_QUESTION,
    pretty_print_prompt,
)

print('✓ Imports complete')

---
### Step 4: Define configs

In [ ]:
# Initialise the shared Config object
# All hyperparameters live here — edit Config in client.py to change models, temperatures, etc.
config = Config()

print('Config:')
print(f'  Decision model:    {config.DECISION_MODEL}')
print(f'  Reflection model:  {config.REFLECTION_MODEL}')
print(f'  Embedding model:   {config.EMBEDDING_MODEL}')
print(f'  Decision temp:     {config.DECISION_TEMPERATURE}')
print(f'  Top-K memories:    {config.TOP_K_MEMORIES} (not used until Stage 2)')

---
### Step 5: Connect to LLM APIs

In [ ]:
# Initialise clients once and reuse throughout (course studio pattern)
# Reads ANTHROPIC_API_KEY and OPENAI_API_KEY from Colab secrets
# Colab: Runtime -> Secrets -> Add ANTHROPIC_API_KEY and OPENAI_API_KEY
client_anthropic, client_openai = init_clients()

---
### Step 6: Load agent config (Margaret)

In [ ]:
# Load Margaret's YAML, derived from real homeowner interview data
AGENT_CONFIG_PATH = Path(PROJECT_PATH) / 'config' / 'agents' / 'margaret.yaml'

with open(AGENT_CONFIG_PATH) as f:
    margaret_config = yaml.safe_load(f)

print(f'Loaded config for: {margaret_config["name"]}')
print(f'  Archetype:         {margaret_config["archetype"]}')
print(f'  Compliance status: {margaret_config["compliance_status"]}')
print(f'  Insurance status:  {margaret_config["insurance_status"]}')
print(f'  Memory seeds:      {len(margaret_config["memory_seeds"])} pre-loaded')
print(f'  Core beliefs:      {len(margaret_config["core_beliefs"])}')

---
### Step 7: Initialise memory stream with seed memories

In [ ]:
margaret_memory = MemoryStream(agent_name='Margaret')

# Load seed memories from YAML
# Calls embed() (OpenAI) and score_importance() (Anthropic) for each seed
margaret_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    client_openai=client_openai,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

print(f'Memory stream ready: {margaret_memory.count()} memories')

In [ ]:
margaret_memory.pretty_print()

---
### Step 8: Define the hardcoded event

In [ ]:
# Hardcoded event - in Stage 3 this will come from scheduler.py reading a YAML scenario file
EVENT = {
    'day': 14,
    'type': 'insurance_non_renewal',
    'channel': 'official_mail',
    'target_agent': 'Margaret',
    'content': (
        'Dear Policyholder, we regret to inform you that your homeowners insurance policy '
        '(Policy #CA-8842-1129) will not be renewed upon its expiration date of June 30, 2026. '
        'This decision is based on our assessment of elevated wildfire risk in your area '
        '(Berkeley Hills, ZIP 94705) following recent catastrophic loss events in California. '
        'We recommend you contact your broker immediately to explore alternative coverage options, '
        'including the California FAIR Plan. You have 60 days to secure alternative coverage.'
    )
}

print(f'Event:   {EVENT["type"]}')
print(f'Channel: {EVENT["channel"]}')
print(f'Day:     {EVENT["day"]}')
print(f'Content: {EVENT["content"][:100]}...')

---
### Step 9: Assemble and inspect the prompt

In [ ]:
# Stage 1: pass ALL seed memories to the LLM
# Stage 2 replaces this with retrieval.py (recency x importance x relevance scoring)
all_memories = margaret_memory.get_all()

situation = frame_event(channel=EVENT['channel'], content=EVENT['content'])

system_prompt = build_system_prompt(
    seed_narrative=margaret_config['seed_narrative'],
    retrieved_memories=all_memories,
)

user_prompt = build_decision_prompt(
    situation=situation,
    decision_question=DECISION_QUESTION,
)

print(f'System prompt: {len(system_prompt):,} chars')
print(f'User prompt:   {len(user_prompt):,} chars')

In [ ]:
# Inspect the full prompt before calling the LLM
pretty_print_prompt(system_prompt, user_prompt)

---
### Step 10: Run the decision pipeline (Perceive -> Retrieve -> Decide)

In [ ]:
print(f'Calling {config.DECISION_MODEL}...\n')

decision_text = decide(
    client_anthropic=client_anthropic,
    system_prompt=system_prompt,
    user_prompt=user_prompt,
)

print('=' * 70)
print('MARGARET\'S DECISION')
print('=' * 70)
print(decision_text)
print('=' * 70)

---
### Step 11: Store the decision as new memories

In [ ]:
# Store the event as an observation memory
obs_description = (
    f'I received an official letter from my insurance company on day {EVENT["day"]} '
    f'informing me that my homeowners policy will not be renewed due to wildfire risk '
    f'in the Berkeley Hills.'
)

obs_importance = score_importance(client_anthropic, obs_description, margaret_config['seed_narrative'])
obs_embedding  = embed(client_openai, obs_description)

margaret_memory.add(
    description=obs_description,
    importance=obs_importance,
    embedding=obs_embedding,
    memory_type='observation',
    timestamp=EVENT['day'],
)
print(f'Stored observation memory (importance: {obs_importance}/10)')

# Summarise and store the decision as a decision memory
summary_prompt = f'Summarise the following decision in one sentence, starting with I decided to...:\n\n{decision_text}'

decision_summary = decide(
    client_anthropic=client_anthropic,
    system_prompt=margaret_config['seed_narrative'],
    user_prompt=summary_prompt,
)

dec_importance = score_importance(client_anthropic, decision_summary, margaret_config['seed_narrative'])
dec_embedding  = embed(client_openai, decision_summary)

margaret_memory.add(
    description=decision_summary,
    importance=dec_importance,
    embedding=dec_embedding,
    memory_type='decision',
    timestamp=EVENT['day'],
)
print(f'Stored decision memory  (importance: {dec_importance}/10)')
print(f'  Summary: {decision_summary}')

In [ ]:
margaret_memory.pretty_print()

---
### Step 12: Log to JSONL

In [ ]:
output_dir = Path(PROJECT_PATH) / 'outputs' / 'runs'
output_dir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = output_dir / f'stage1_margaret_{run_id}.jsonl'

log_record = {
    'run_id': run_id,
    'stage': 1,
    'agent': margaret_config['name'],
    'event': EVENT,
    'framed_situation': situation,
    'decision_text': decision_text,
    'decision_summary': decision_summary,
    'memory_stream_after': [m.to_dict() for m in margaret_memory.get_all()],
}

with open(output_path, 'w') as f:
    f.write(json.dumps(log_record) + '\n')

print(f'Logged run to: {output_path}')

---
### Step 13: Evaluation — does Margaret sound like Margaret?

This is the core Stage 1 test. We compare the simulated response against Margaret's real held-out interview response.

**Approach:** LLM-as-judge scoring, adapted from `judge_with_llm()` in course notebook `2_Full_MRAG_w_ColPali_HF.ipynb`.

**Signals to look for in Margaret's response:**
- Does she mention her brick patio or compliance work as something to document?
- Does she want to understand the insurer's specific criteria?
- Does she contact neighbors or her Firewise group?
- Does she research alternative insurance (FAIR Plan, brokers)?
- Does she frame it as unfair / an advocacy opportunity?
- She should NOT immediately panic and give up (she is determined)
- She should NOT ignore it (she is an information-seeker)

In [ ]:
print('=' * 70)
print('HELD-OUT REAL RESPONSE (from Margaret interview):')
print('=' * 70)
print(margaret_config['held_out_responses']['insurance_non_renewal']['real_response'])
print()
print('=' * 70)
print('SIMULATED RESPONSE (from LLM):')
print('=' * 70)
print(decision_text)

In [ ]:
# LLM-as-judge scoring
# Adapted from judge_with_llm() in 2_Full_MRAG_w_ColPali_HF.ipynb (Cornelia Paulik, INFO 290)

def judge_persona_fidelity(client_anthropic, real_response, simulated_response, agent_name, max_retries=3):
    prompt = f"""You are evaluating a multi-agent simulation of wildfire mitigation decision-making.

Agent: {agent_name}
Real interview response: {real_response}
Simulated response: {simulated_response}

Score the simulated response on these 2 criteria, each on a scale of 1 to 5:
- behavioral_fidelity: Does the simulated agent make the same decisions and take the same actions as the real person described?
- tone_fidelity: Does the simulated agent's emotional tone and priorities match the real person's?

Respond ONLY with valid JSON in this exact format (no preamble, no markdown):
{{"behavioral_fidelity_score": <int>, "behavioral_fidelity_reason": "<str>", "tone_fidelity_score": <int>, "tone_fidelity_reason": "<str>"}}
"""
    for attempt in range(max_retries):
        message = client_anthropic.messages.create(
            model=config.DECISION_MODEL,
            max_tokens=512,
            temperature=0.0,
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = message.content[0].text.strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f'[judge] Attempt {attempt+1}: JSON parse failed, retrying...')
    return {'error': f'Failed after {max_retries} attempts', 'raw': raw}


scores = judge_persona_fidelity(
    client_anthropic=client_anthropic,
    real_response=margaret_config['held_out_responses']['insurance_non_renewal']['real_response'],
    simulated_response=decision_text,
    agent_name='Margaret',
)

print('=' * 70)
print('LLM-AS-JUDGE SCORES')
print('=' * 70)
print(f"Behavioral fidelity: {scores.get('behavioral_fidelity_score', 'N/A')}/5")
print(f"  Reason: {scores.get('behavioral_fidelity_reason', 'N/A')}")
print()
print(f"Tone fidelity: {scores.get('tone_fidelity_score', 'N/A')}/5")
print(f"  Reason: {scores.get('tone_fidelity_reason', 'N/A')}")

In [ ]:
# Quick re-run helper — tune seed_narrative in margaret.yaml or DECISION_QUESTION in prompts.py,
# then call this without reloading seed memories (saves API cost)

def quick_rerun(new_seed_narrative=None):
    seed = new_seed_narrative or margaret_config['seed_narrative']
    sys_prompt = build_system_prompt(seed_narrative=seed, retrieved_memories=margaret_memory.get_all())
    usr_prompt = build_decision_prompt(situation=frame_event(EVENT['channel'], EVENT['content']), decision_question=DECISION_QUESTION)
    result = decide(client_anthropic, sys_prompt, usr_prompt)
    print('=' * 70)
    print('MARGARET\'S DECISION (re-run)')
    print('=' * 70)
    print(result)
    return result

# Uncomment to run:
# quick_rerun()

---
## Stage 1 Complete

**What we built:**
- `config/agents/margaret.yaml` — Margaret's persona, seed memories, held-out interview responses
- `src/llm/client.py` — Config dataclass + API calls (decide, embed, score_importance, reflect)
- `src/agents/memory.py` — Append-only MemoryStream with Memory dataclass
- `src/agents/prompts.py` — Prompt assembly (seed + memories + situation + decision question)

**What Stage 2 adds (`retrieval.py`):**
- Score memories by recency (alpha) x importance (beta) x relevance (gamma), return top-K
- Feed Margaret 5-10 sequential events and check which memories surface at each decision point
- Tune alpha, beta, gamma weights interactively

**Tuning checklist before moving to Stage 2:**
- [ ] Behavioral fidelity score >= 3/5
- [ ] Tone fidelity score >= 3/5
- [ ] Margaret mentions her brick patio / compliance work
- [ ] Margaret researches the insurer's specific criteria
- [ ] Margaret contacts neighbors or Firewise group